# TTE map review

Maps TTE map tiles that DPS (`run_tte_map`) published in the MAAP DPS STAC catalog, through the MAAP
TiTiler. Four steps:

1. **What's published**: a table of every run so far, one row per collection and run label
   (`tte:tcc_label`, e.g. `hansen2010`, `metachmv2_2m`).
2. **Choose** the runs to map in `SHOW` (and optionally a region).
3. **Map**: each chosen run gets its own layers in the layer list (TTE classes, tree cover,
   abruptness, tile grid). **Hover** a tile for its metadata; **click** a pixel for its class, tree
   cover, abruptness and zone in every chosen run, side by side.
4. **Update the web page** (only when you want to): the published review map,
   https://pahbs.github.io/tte/docs/tte_review_map.html. Steps 1-3 do not change it.

Run the cells yourself in a Hub kernel: JupyterLab only draws the map from output you created.
Zoomed out to continental scale (below zoom 5), the TTE classes and tree cover come from a 1 km
**overview** of each run (one small COG; TiTiler is too slow to mosaic thousands of tiles at those
zooms). Step 3 builds or updates the overviews; zoomed in, the layers switch to the 30 m tiles.

In [6]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'lib'))   # ../lib (run from notebooks/)
sys.path.insert(0, os.path.expanduser('~/code/tte/lib'))
import pandas as pd
import tte_review as tr
pd.set_option('display.max_colwidth', 60)

## 1. What's published

One row per run: the DPS STAC collection (MAAP names it `{user}__<algorithm>_<n>__{version}`; the
`_<n>` is added by MAAP), the run label, the tree cover it was made with, how many tiles, where they are
(`extent`: W, S, E, N in degrees), the DPS job tags and when the tiles were made.

In [7]:
runs = tr.published_runs('montesano', 'run_tte_map')
runs

,collection,version,run,tcc_source,tcc_year,tcc_options,tiles,extent,job_tags,first_created,last_created
0,montesano__run_tte_map_13__tte_map_v1.1,tte_map_v1.1,metachmv2_2m,meta_chm,2020,ht_min=2;version=v2,1479,"-179, 48, 180, 74","tte_map_metachmv2_2m, tte_queue_test_8gb",2026-11-24T01:56,2026-11-25T22:00
1,montesano__run_tte_map_13__tte_map_v1.1,tte_map_v1.1,hansen2010,hansen,2010,version=GFC-2018-v1.6,1,"-161, 67, -160, 68",tte_map_hansen2010,2026-11-24T01:36,2026-11-24T01:36


## 2. Choose

* `SHOW`: the runs to map, as `(collection, run)` pairs from the table. It starts with every run in
  the table; delete the ones you don't want. The map opens on the first run, with its classes switched
  on; the others are in the layer list (runs in other regions: pan there, or put that run first).
* `REGION`: `None` for all published tiles of those runs, or a lon/lat box `(W, S, E, N)`, e.g. one of
  `REGIONS`.
* `REFERENCE`: your original c2010 map (GEE `TTE_DATA_v201902v2`). Its classes come from its 30 m file
  (`lib/tte_display.py`), its tree cover from its 1 km overview (`lib/tte_overview.py`). Once it is published
  in the DPS STAC (`dps/DPS_tte_map.ipynb`, section 7) its values are also in the click popup. `None` leaves it out.

In [8]:
SHOW = list(zip(runs.collection, runs.run))      # edit: keep the (collection, run) pairs to map
REGIONS = {'western Alaska': (-166.0, 63.5, -155.0, 69.0), 'Brooks Range': (-155.0, 66.5, -141.0, 69.0)}
REGION = None                                    # e.g. REGIONS['Brooks Range']
REFERENCE = tr.find_reference()                  # the original c2010 map (GEE TTE_DATA_v201902v2); None = leave it out
print('reference:', REFERENCE or 'none yet (tte_overview.build_reference_overview(); dps/DPS_tte_map.ipynb, section 7)')

for s in SHOW:
    print(s)

reference: {'overview': 's3://maap-ops-workspace/montesano/dps_output/tte_overviews/reference_c2010_gee_tte_data_v201902v2.tif', 'display': 's3://maap-ops-workspace/montesano/dps_output/tte_overviews/display_reference_c2010_gee_tte_data_v201902v2_nearest.tif'}
('montesano__run_tte_map_13__tte_map_v1.1', 'metachmv2_2m')
('montesano__run_tte_map_13__tte_map_v1.1', 'hansen2010')


## 3. Map

`tr.build_overviews(SHOW)` builds the 1 km overview of each run that has none, or that has fewer tiles
than the run has now (about 0.3 s per tile with 12 threads, so a few minutes for thousands of tiles).

In [9]:
tr.build_overviews(SHOW)
m = tr.review_map(SHOW, bbox=REGION, reference=REFERENCE)
m

metachmv2_2m: overview has 1471 tiles, the run has 1479: building
montesano__run_tte_map_13__tte_map_v1.1 / metachmv2_2m: 1479 tiles
  200/1479 tiles (9s)
  400/1479 tiles (14s)
  600/1479 tiles (21s)
  800/1479 tiles (28s)
  1000/1479 tiles (35s)
  1200/1479 tiles (42s)
  1400/1479 tiles (47s)
montesano__run_tte_map_13__tte_map_v1.1__metachmv2_2m: 1479 tiles -> /home/jovyan/my-private-bucket/dps_output/tte_overviews/montesano__run_tte_map_13__tte_map_v1.1__metachmv2_2m.tif (55s)
hansen2010: overview up to date (1 tiles)
metachmv2_2m: 1479 tiles from montesano__run_tte_map_13__tte_map_v1.1
hansen2010: 1 tiles from montesano__run_tte_map_13__tte_map_v1.1
Reference c2010 (GEE): 1 km overview (at every zoom; full resolution once published, see dps/DPS_tte_map.ipynb section 7)


## 4. Update the web page

The page https://pahbs.github.io/tte/docs/tte_review_map.html shows the Meta run from one file, made from the tiles
published when it was built. To include the tiles published since, set `UPDATE_PAGE = True` and run this cell:

1. `build_run` joins the run's published tiles into a new 30 m file (about 10 minutes for 1,200 tiles, longer for the full set).
2. `update_page` writes that file and the tile count into `docs/tte_review_map.html`.

The live page changes after `docs/tte_review_map.html` is committed and pushed. The c2010 map never needs updating.

In [11]:
UPDATE_PAGE = False                              # True: rebuild the run's file and update the page
PAGE_RUN = ('montesano__run_tte_map_13__tte_map_v1.1', 'metachmv2_2m')      # (collection, run) on the page

if UPDATE_PAGE:
    import tte_display as td
    td.build_run(*PAGE_RUN)
    td.update_page(*PAGE_RUN)
else:
    print('page not updated: set UPDATE_PAGE = True')

display_montesano__run_tte_map_13__tte_map_v1.1__metachmv2_2m_20261125T2212: 1489 tiles, 220396 x 244042, grid read in 60s
  48/982 chunks (77s)
  528/982 chunks (404s)
  982/982 chunks (692s)
  overviews (728s)
  /home/jovyan/my-private-bucket/dps_output/tte_overviews/display_montesano__run_tte_map_13__tte_map_v1.1__metachmv2_2m_20261125T2212.tif: 1.29 GB (803s)
/home/jovyan/code/tte/docs/tte_review_map.html 1,450 of 3,565 tiles as of 2026-11-25
